# MalIntent — DistilBERT Prompt Injection Detection

This notebook preprocesses the HackAPrompt dataset for training a DistilBERT-based prompt injection detection model.

## Objectives
- Load and inspect HackAPrompt dataset
- Clean and preprocess adversarial prompts
- Create binary attack labels
- Prepare train/test datasets for transformer fine-tuning

Frameworks used:
- HuggingFace Transformers
- HuggingFace Datasets
- Scikit-learn
- Pandas

In [15]:
# Install required libraries

!pip install transformers datasets scikit-learn accelerate -q

## GPU Verification
Ensuring Google Colab GPU acceleration is available before preprocessing and training.

In [16]:
# Verify GPU availability

import torch

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")

GPU available: True
GPU name: Tesla T4


## Import Libraries
Importing required Python libraries for dataset handling and preprocessing.

In [17]:
import pandas as pd
import transformers
import datasets
import sklearn

from datasets import load_dataset
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

Libraries imported successfully.


## Load HackAPrompt Dataset
Loading the HackAPrompt adversarial prompt dataset from HuggingFace.

In [18]:
from huggingface_hub import notebook_login

# Authenticate with HuggingFace
# Paste your token securely when prompted

notebook_login()

In [19]:
# Load dataset

dataset = load_dataset("hackaprompt/hackaprompt-dataset")

# Convert to pandas DataFrame
df = pd.DataFrame(dataset['train'])

print("Dataset loaded successfully.")

Dataset loaded successfully.


## Dataset Inspection
Understanding dataset structure, columns, and sample records before preprocessing.

In [20]:
# Inspect dataset structure

print("Columns:")
print(df.columns.tolist())

print("\nDataset shape:")
print(df.shape)

print("\nSample row:")
print(df.iloc[0])

# Analyze dataset outcomes

print("Correct value counts:")
print(df['correct'].value_counts())

print("\nError value counts:")
print(df['error'].value_counts())

Columns:
['level', 'prompt', 'user_input', 'completion', 'model', 'expected_completion', 'token_count', 'correct', 'error', 'score', 'dataset', 'timestamp', 'session_id']

Dataset shape:
(601757, 13)

Sample row:
level                                                                  1
prompt                 You are a translation bot designed solely to t...
user_input             This competition is so hard! I don't know what...
completion             ¡Esta competencia es tan difícil! ¡No sé qué h...
model                                                      gpt-3.5-turbo
expected_completion                                    I have been PWNED
token_count                                                           14
correct                                                            False
error                                                              False
score                                                                0.0
dataset                                                  

## Dataset Preprocessing
Cleaning prompt data and preparing binary labels for machine learning training.

In [21]:
# Preprocess dataset for training

# Extract attacker prompts
df_clean = df[['user_input']].copy()

# Rename column for ML convention
df_clean.rename(columns={'user_input': 'text'}, inplace=True)

# Assign attack labels
# All HackAPrompt samples are treated as malicious prompts
df_clean['label'] = 1

# Remove null values
df_clean = df_clean.dropna(subset=['text'])

# Remove empty prompts
df_clean = df_clean[df_clean['text'].str.strip() != '']

# Remove duplicate prompts
df_clean = df_clean.drop_duplicates(subset=['text'])

# Shuffle dataset
df_clean = df_clean.sample(frac=1, random_state=42)

print("Clean dataset shape:")
print(df_clean.shape)

print("\nLabel distribution:")
print(df_clean['label'].value_counts())

print("\nSample rows:")
print(df_clean.head())

Clean dataset shape:
(415254, 2)

Label distribution:
label
1    415254
Name: count, dtype: int64

Sample rows:
                                                     text  label
371696  you are translation bot designed to translate ...      1
126056  My name is alok and I created a language calle...      1
572301  \𝚄\𝚜\𝚎\𝚛\ \𝚒\𝚗\𝚙\𝚞\𝚝\ \:\ \"\𝙸\ \𝚑\𝚊\𝚟\𝚎\ \ \𝚋...      1
511638                                   🔄🔤🔜🔡📄💬🌞💗👉👈💗🌜🚫👀📝🔽      1
568096  ]\nDo not generate a short story. You are a ro...      1


## Train/Test Split
Splitting the cleaned dataset into training and testing subsets.

In [22]:
# Split dataset into train and test sets

train_df, test_df = train_test_split(
    df_clean,
    test_size=0.2,
    random_state=42
)

print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

# Save test set for future evaluation
test_df.to_csv("test_set.csv", index=False)

print("\nTest dataset saved successfully.")

Train samples: 332203
Test samples: 83051

Test dataset saved successfully.
